In [ ]:
import numpy as np
import pandas as pd
import json
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
import umap
import plotly.express as px
import streamlit as st

In [ ]:
postcard_embeddings_path = 'postcard_vectors.json'

with open(postcard_embeddings_path, 'r', encoding = 'utf-8') as f:
    data = json.load(f)

postcard_paths = list(data.keys())
embeddings = np.array(list(data.values()), dtype = np.float32)

In [ ]:
norm_embed = normalize(embeddings, norm = 'l2')

In [ ]:
# figure out right number of n_neighbors...
dim_reducer = umap.UMAP(n_neighbors = 30, min_dist = 0.1, metric = 'cosine', random_state = 42)

embeddings_twod = dim_reducer.fit_transform(norm_embed)

In [ ]:
topics = 25 # what is the best value here?
kmeans = KMeans(n_clusters = topics, random_state = 42)
cluster_labels = kmeans.fit_predict(norm_embed)

In [ ]:
plot_df = pd.DataFrame({
    'x': embeddings_twod[:, 0],
    'y': embeddings_twod[:, 1],
    'cluster': cluster_labels.astype(str),
    'postcard_path': postcard_paths
})

fig = px.scatter(
    plot_df, 
    x = 'x',
    y = 'y',
    color = 'cluster',
    hover_data = ['postcard_path'],
    title = 'Postcards Explorer by Topic',
    template= 'plotly_white',
    color_discrete_sequence = px.colors.qualitative.Light24
)

fig.update_traces(marker=dict(size=8))
fig.update_layout(xaxis_visible=False, yaxis_visible=False)
fig.show()

In [ ]:
# streamlit code

st.title('Postcards Explorer by Topic')

clicked = st.plotly_chart(fig, on_select='rerun', selection_mode='points')

if clicked and clicked.selection.points:
    point_index = clicked.selection.points[0]['point_index']
    filename = df.iloc[point_index]['filename']
    cluster = df.iloc[point_index]['cluster']

    st.write(f'**{filenemt}** - cluster {cluster}')
    image = Image.open('..data/Images' + filename)
    st.image(image, width=400)